In [1]:
from load_data import load_raw_data
from clean_data import clean_data
from split_data import split_data
from evaluate_model import evaluate_model
from config import RANDOM_STATE
from build_pipeline import objective,build_pipeline
import optuna
import logging

In [2]:
logging.basicConfig(
    level=logging.INFO,format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")

In [3]:
# Loading raw data
dataset=load_raw_data()

2026-06-05 00:03:29,614 | INFO | load_data | Raw dataset loaded.


In [4]:
# Cleaning data
dataset_cleaned=clean_data(dataset)

2026-06-05 00:03:29,619 | INFO | clean_data | Dropped customerID.
2026-06-05 00:03:29,621 | INFO | clean_data | Converted TotalCharges to numeric.
2026-06-05 00:03:29,626 | INFO | clean_data | Rows containing NaN before cleaning: 11
2026-06-05 00:03:29,634 | INFO | clean_data | Removed 11 rows with invalid TotalCharges. Remaining rows with NaN: 0.


In [5]:
# Splitting data
X_train, X_test, y_train, y_test=split_data(dataset_cleaned, random_state=RANDOM_STATE)

2026-06-05 00:03:29,641 | INFO | split_data | Data split into train/test set with 0.8/0.2 proportion and random_state=0.


### XGBoost

In [6]:
# Creating optuna study for XGBoost
study_XGB = optuna.create_study(direction="maximize")

[I 2026-06-04 23:48:56,569] A new study created in memory with name: no-name-1c948601-2599-437b-89f1-81b413994b88


In [7]:
# Optimizing optuna study for XGBoost
study_XGB.optimize(lambda trial: objective(trial, X_train, y_train, model='XGBoost'),n_trials=100)

[I 2026-06-04 23:48:59,182] Trial 0 finished with value: 0.5299767066074667 and parameters: {'xgb_learning_rate': 0.001188511213588468, 'xgb_max_depth': 6, 'xgb_min_child_weight': 3.1407365712968165, 'xgb_gamma': 2.450737685229234, 'xgb_subsample': 0.8229233711334011, 'xgb_colsample_bytree': 0.7636851267443042, 'xgb_colsample_bylevel': 0.5576273417231796, 'xgb_reg_alpha': 1.1858193659478827e-08, 'xgb_reg_lambda': 0.955268890315717, 'xgb_scale_pos_weight': 16.58034319493985}. Best is trial 0 with value: 0.5299767066074667.
[I 2026-06-04 23:49:01,021] Trial 1 finished with value: 0.5436379530977923 and parameters: {'xgb_learning_rate': 0.002770260028477133, 'xgb_max_depth': 5, 'xgb_min_child_weight': 2.465503012818593, 'xgb_gamma': 7.739833080472883, 'xgb_subsample': 0.8311079481380842, 'xgb_colsample_bytree': 0.7428998135452176, 'xgb_colsample_bylevel': 0.5163402454671228, 'xgb_reg_alpha': 3.5008814416167176e-06, 'xgb_reg_lambda': 0.054946068906880995, 'xgb_scale_pos_weight': 16.2422587

In [8]:
print("Best score for XGB:", study_XGB.best_value)
print("Best params for XGB:", study_XGB.best_params)

Best score for XGB: 0.6371090172277298
Best params for XGB: {'xgb_learning_rate': 0.00972281300535932, 'xgb_max_depth': 3, 'xgb_min_child_weight': 19.982211758715323, 'xgb_gamma': 7.955115758314879, 'xgb_subsample': 0.9237921475751399, 'xgb_colsample_bytree': 0.8888048464270398, 'xgb_colsample_bylevel': 0.7794163552951257, 'xgb_reg_alpha': 5.8423907850128914e-05, 'xgb_reg_lambda': 7.174417029795075e-07, 'xgb_scale_pos_weight': 2.069962871037832}


In [9]:
# Fitting Logistic Regression pipeline with the best trial
pipe_XGB=build_pipeline(trial=study_XGB.best_trial, model= 'XGBoost')
fitted_pipe_XGB=pipe_XGB.fit(X_train, y_train)

### LightGBM

In [6]:
# Creating optuna study for LightGBM
study_LGBM = optuna.create_study(direction="maximize")

[I 2026-06-05 00:03:31,856] A new study created in memory with name: no-name-6868c481-b484-4626-ad23-9fd1d3fe3dce


In [7]:
# Optimizing optuna study for LightGBM
study_LGBM.optimize(lambda trial: objective(trial, X_train, y_train, model='LightGBM'),n_trials=100)

[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.189896 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.243208 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.149981 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

...

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-06-05 00:08:20,458] Trial 2 finished with value: 0.5973863389848756 and parameters: {'lgbm_learning_rate': 0.025135320877542732, 'lgbm_max_depth': 10, 'lgbm_num_leaves': 503, 'lgbm_min_child_samples': 71, 'lgbm_subsample': 0.9678657543702787, 'lgbm_colsample_bytree': 0.9249253494381747, 'lgbm_reg_alpha': 5.266486392238566, 'lgbm_reg_lambda

[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000227 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

[W 2026-06-05 00:08:28,702] Trial 3 failed with parameters: {'lgbm_learning_rate': 0.015865336698271375, 'lgbm_max_depth': 7, 'lgbm_num_leaves': 84, 'lgbm_min_child_samples': 41, 'lgbm_subsample': 0.8305370358277393, 'lgbm_colsample_bytree': 0.744038854750514, 'lgbm_reg_alpha': 0.04898642704240225, 'lgbm_reg_lambda': 0.18064302986657754} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/joblib/parallel.py", line 1682, in _get_outputs
    yield from self._retrieve()
  File "/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/joblib/parallel.py", line 1800, in _retrieve
    time.sleep(0.01)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func

In [ ]:
print("Best score for LGBM:", study_LGBM.best_value)
print("Best params for LGBM:", study_LGBM.best_params)

In [ ]:
# Fitting LightGBM pipeline with the best trial
pipe_LGBM=build_pipeline(trial=study_LR.best_trial, model= 'LightGBM')
fitted_pipe_LGBM=pipe_LGBM.fit(X_train, y_train)

### Logistic Regression

In [6]:
# Creating optuna study for Logistic Regression
study_LR = optuna.create_study(direction="maximize")

[I 2026-06-04 13:19:02,448] A new study created in memory with name: no-name-546b40e1-ca68-4b09-be98-cacc2772162d


In [7]:
# Optimizing optuna study for Logistic Regression
study_LR.optimize(lambda trial: objective(trial, X_train, y_train, model='Logistic Regression'),n_trials=100)

[I 2026-06-04 13:19:07,108] Trial 0 finished with value: 0.6314416638549953 and parameters: {'solver': 'saga', 'l1_ratio': 0.8225903096916659, 'C': 23.721294518237997}. Best is trial 0 with value: 0.6314416638549953.
[I 2026-06-04 13:19:08,721] Trial 1 finished with value: 0.6317913229076122 and parameters: {'solver': 'saga', 'l1_ratio': 0.7071713408245592, 'C': 785.888448659899}. Best is trial 1 with value: 0.6317913229076122.
[I 2026-06-04 13:19:11,092] Trial 2 finished with value: 0.6305924644680453 and parameters: {'solver': 'saga', 'l1_ratio': 0.5947445327336515, 'C': 3.8339597065703477}. Best is trial 1 with value: 0.6317913229076122.
[I 2026-06-04 13:19:13,193] Trial 3 finished with value: 0.6307589935315828 and parameters: {'solver': 'saga', 'l1_ratio': 0.9015894359326714, 'C': 7.271756254288223}. Best is trial 1 with value: 0.6317913229076122.
[I 2026-06-04 13:19:14,399] Trial 4 finished with value: 0.6287909791992863 and parameters: {'solver': 'saga', 'l1_ratio': 0.1424208738

In [8]:
print("Best score for LR:", study_LR.best_value)
print("Best params for LR:", study_LR.best_params)

Best score for LR: 0.632291841475064
Best params for LR: {'solver': 'saga', 'l1_ratio': 0.2508065392777322, 'C': 33.77695307075628}


In [9]:
# Fitting Logistic Regression pipeline with the best trial
pipe_LR=build_pipeline(trial=study_LR.best_trial, model= 'Logistic Regression')
fitted_pipe_LR=pipe_LR.fit(X_train, y_train)

### K-Nearest Neighbors

In [11]:
# Creating optuna study for K-Nearest Neighbors
study_KNN = optuna.create_study(direction="maximize")

[I 2026-06-04 13:21:58,060] A new study created in memory with name: no-name-2b6fc4b9-6ff5-47cb-8065-2dd5bd9c4d66


In [12]:
# Optimizing optuna study for K-Nearest Neighbors
study_KNN.optimize(lambda trial: objective(trial, X_train, y_train, model='K-Nearest Neighbors'),n_trials=100)

[I 2026-06-04 13:21:58,791] Trial 0 finished with value: 0.581352462535277 and parameters: {'knn_n_neighbors': 43, 'knn_weights': 'distance', 'knn_metric': 'manhattan', 'knn_p': 1}. Best is trial 0 with value: 0.581352462535277.
[I 2026-06-04 13:21:58,908] Trial 1 finished with value: 0.5791216292395749 and parameters: {'knn_n_neighbors': 17, 'knn_weights': 'uniform', 'knn_metric': 'euclidean', 'knn_p': 1}. Best is trial 0 with value: 0.581352462535277.
[I 2026-06-04 13:22:00,836] Trial 2 finished with value: 0.578327888648321 and parameters: {'knn_n_neighbors': 25, 'knn_weights': 'distance', 'knn_metric': 'minkowski', 'knn_p': 3}. Best is trial 0 with value: 0.581352462535277.
[I 2026-06-04 13:22:01,084] Trial 3 finished with value: 0.5950548161240115 and parameters: {'knn_n_neighbors': 31, 'knn_weights': 'uniform', 'knn_metric': 'manhattan', 'knn_p': 3}. Best is trial 3 with value: 0.5950548161240115.
[I 2026-06-04 13:22:01,191] Trial 4 finished with value: 0.5772403884155576 and par

In [13]:
print("Best score for KNN:", study_KNN.best_value)
print("Best params for KNN:", study_KNN.best_params)

Best score for KNN: 0.6031276199829867
Best params for KNN: {'knn_n_neighbors': 47, 'knn_weights': 'uniform', 'knn_metric': 'manhattan', 'knn_p': 2}


In [14]:
# Fitting K-Nearest Neighbors pipeline with the best trial
pipe_KNN=build_pipeline(trial=study_KNN.best_trial, model= 'K-Nearest Neighbors')
fitted_pipe_KNN=pipe_KNN.fit(X_train, y_train)

### Support Vector Machine

In [6]:
# Creating optuna study for Support Vector Machine
study_SVM = optuna.create_study(direction="maximize")

[I 2026-05-26 23:39:13,045] A new study created in memory with name: no-name-cc97c002-270c-45bc-acf6-52a72cf87b03


In [7]:
# Optimizing optuna study for Support Vector Machine
study_SVM.optimize(lambda trial: objective(trial, X_train, y_train, model='Support Vector Machine'),n_trials=100)

[I 2026-05-26 23:39:16,486] Trial 0 finished with value: 0.7342222222222222 and parameters: {'svc_C': 0.2339226625915227, 'svc_kernel': 'poly', 'svc_gamma': 1.956220607680727e-05, 'svc_degree': 2}. Best is trial 0 with value: 0.7342222222222222.
[I 2026-05-26 23:39:19,382] Trial 1 finished with value: 0.7962666666666667 and parameters: {'svc_C': 0.25598274400210363, 'svc_kernel': 'linear', 'svc_gamma': 4.597426790817768, 'svc_degree': 3}. Best is trial 1 with value: 0.7962666666666667.
[I 2026-05-26 23:39:26,344] Trial 2 finished with value: 0.7646222222222222 and parameters: {'svc_C': 0.012491066662790104, 'svc_kernel': 'poly', 'svc_gamma': 1.126868090051555, 'svc_degree': 3}. Best is trial 1 with value: 0.7962666666666667.
[I 2026-05-26 23:45:06,143] Trial 3 finished with value: 0.6991999999999999 and parameters: {'svc_C': 0.030476830970179634, 'svc_kernel': 'poly', 'svc_gamma': 5.348052285040963, 'svc_degree': 5}. Best is trial 1 with value: 0.7962666666666667.
[I 2026-05-26 23:45:0

In [8]:
print("Best score for SVM:", study_SVM.best_value)
print("Best params for SVM:", study_SVM.best_params)

Best score for SVM: 0.8026666666666668
Best params for SVM: {'svc_C': 99.47143262177474, 'svc_kernel': 'rbf', 'svc_gamma': 0.003346123875332518, 'svc_degree': 5}


In [9]:
# Fitting Support Vector Machine pipeline with the best trial
pipe_SVM=build_pipeline(trial=study_SVM.best_trial, model= 'Support Vector Machine')
fitted_pipe_SVM=pipe_SVM.fit(X_train, y_train)

### Decision Tree

In [16]:
# Creating optuna study for Decision Tree
study_DT = optuna.create_study(direction="maximize")

[I 2026-06-04 13:23:32,412] A new study created in memory with name: no-name-57666840-d52c-4170-8968-e65b730d9b2f


In [17]:
# Optimizing optuna study for Decision Tree
study_DT.optimize(lambda trial: objective(trial, X_train, y_train, model='Decision Tree'),n_trials=100)

[I 2026-06-04 13:23:32,874] Trial 0 finished with value: 0.570902895170087 and parameters: {'dt_criterion': 'entropy', 'dt_max_depth': 26, 'dt_min_samples_split': 8, 'dt_min_samples_leaf': 9, 'dt_max_features': None}. Best is trial 0 with value: 0.570902895170087.
[I 2026-06-04 13:23:32,941] Trial 1 finished with value: 0.5782581782293459 and parameters: {'dt_criterion': 'entropy', 'dt_max_depth': 21, 'dt_min_samples_split': 15, 'dt_min_samples_leaf': 2, 'dt_max_features': 'log2'}. Best is trial 1 with value: 0.5782581782293459.
[I 2026-06-04 13:23:33,029] Trial 2 finished with value: 0.570902895170087 and parameters: {'dt_criterion': 'entropy', 'dt_max_depth': 27, 'dt_min_samples_split': 16, 'dt_min_samples_leaf': 9, 'dt_max_features': None}. Best is trial 1 with value: 0.5782581782293459.
[I 2026-06-04 13:23:33,097] Trial 3 finished with value: 0.5730857131528496 and parameters: {'dt_criterion': 'entropy', 'dt_max_depth': 35, 'dt_min_samples_split': 9, 'dt_min_samples_leaf': 5, 'dt_m

In [18]:
print("Best score for DT:", study_DT.best_value)
print("Best params for DT:", study_DT.best_params)

Best score for DT: 0.6058262001931952
Best params for DT: {'dt_criterion': 'gini', 'dt_max_depth': 7, 'dt_min_samples_split': 20, 'dt_min_samples_leaf': 8, 'dt_max_features': 'sqrt'}


In [19]:
# Fitting Decision Tree pipeline with the best trial
pipe_DT=build_pipeline(trial=study_DT.best_trial, model= 'Decision Tree')
fitted_pipe_DT=pipe_DT.fit(X_train, y_train)

### Random Forest

In [22]:
# Creating optuna study for Random Forest
study_RF = optuna.create_study(direction="maximize")

[I 2026-06-04 13:24:26,147] A new study created in memory with name: no-name-f5711467-f094-4dcb-9a53-c102afa1929c


In [23]:
# Optimizing optuna study for Random Forest
study_RF.optimize(lambda trial: objective(trial, X_train, y_train, model='Random Forest'),n_trials=100)

[I 2026-06-04 13:24:30,653] Trial 0 finished with value: 0.6003063321145069 and parameters: {'rf_n_estimators': 1000, 'rf_criterion': 'gini', 'rf_max_depth': 32, 'rf_min_samples_split': 13, 'rf_min_samples_leaf': 1, 'rf_max_features': None, 'rf_bootstrap': True}. Best is trial 0 with value: 0.6003063321145069.
[I 2026-06-04 13:24:32,834] Trial 1 finished with value: 0.5762352399174422 and parameters: {'rf_n_estimators': 400, 'rf_criterion': 'gini', 'rf_max_depth': 17, 'rf_min_samples_split': 14, 'rf_min_samples_leaf': 9, 'rf_max_features': None, 'rf_bootstrap': False}. Best is trial 0 with value: 0.6003063321145069.
[I 2026-06-04 13:24:33,105] Trial 2 finished with value: 0.6329302032783853 and parameters: {'rf_n_estimators': 100, 'rf_criterion': 'entropy', 'rf_max_depth': 29, 'rf_min_samples_split': 4, 'rf_min_samples_leaf': 9, 'rf_max_features': 'log2', 'rf_bootstrap': False}. Best is trial 2 with value: 0.6329302032783853.
[I 2026-06-04 13:24:33,919] Trial 3 finished with value: 0.6

In [24]:
print("Best score for RF:", study_RF.best_value)
print("Best params for RF:", study_RF.best_params)

Best score for RF: 0.6408071176934772
Best params for RF: {'rf_n_estimators': 500, 'rf_criterion': 'gini', 'rf_max_depth': 16, 'rf_min_samples_split': 17, 'rf_min_samples_leaf': 8, 'rf_max_features': 'log2', 'rf_bootstrap': True}


In [25]:
# Fitting Random Forest pipeline with the best trial
pipe_RF=build_pipeline(trial=study_RF.best_trial, model= 'Random Forest')
fitted_pipe_RF=pipe_RF.fit(X_train, y_train)

### Model performances

In [11]:
# Evaluate model performance
metrics_XGB=evaluate_model(fitted_pipe_XGB,X_test,y_test)
print(metrics_XGB['report'])

2026-06-04 23:51:38,234 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for XGBClassifier classification model.


              precision    recall  f1-score   support

   Not Churn       0.89      0.80      0.84      1033
       Churn       0.57      0.74      0.64       374

    accuracy                           0.78      1407
   macro avg       0.73      0.77      0.74      1407
weighted avg       0.81      0.78      0.79      1407



In [ ]:
metrics_LGBM=evaluate_model(fitted_pipe_LGBM,X_test,y_test)
print(metrics_LGBM['report'])

[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.166663 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [10]:
metrics_LR=evaluate_model(fitted_pipe_LR,X_test,y_test)
print(metrics_LR['report'])

2026-06-04 13:21:41,529 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for LogisticRegression classification model.


              precision    recall  f1-score   support

   Not Churn       0.91      0.74      0.81      1033
       Churn       0.52      0.79      0.63       374

    accuracy                           0.75      1407
   macro avg       0.72      0.77      0.72      1407
weighted avg       0.81      0.75      0.77      1407



In [15]:
metrics_KNN=evaluate_model(fitted_pipe_KNN,X_test,y_test)
print(metrics_KNN['report'])

2026-06-04 13:22:21,803 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for KNeighborsClassifier classification model.


              precision    recall  f1-score   support

   Not Churn       0.86      0.88      0.87      1033
       Churn       0.64      0.60      0.62       374

    accuracy                           0.81      1407
   macro avg       0.75      0.74      0.75      1407
weighted avg       0.80      0.81      0.80      1407



In [10]:
metrics_SVM=evaluate_model(fitted_pipe_SVM,X_test,y_test)
print(metrics_SVM['report'])

2026-05-26 23:57:40,727 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for SVC classification model.


              precision    recall  f1-score   support

   Not Churn       0.84      0.93      0.88      1033
       Churn       0.71      0.49      0.58       374

    accuracy                           0.81      1407
   macro avg       0.77      0.71      0.73      1407
weighted avg       0.80      0.81      0.80      1407



In [21]:
metrics_DT=evaluate_model(fitted_pipe_DT,X_test,y_test)
print(metrics_DT['report'])

2026-06-04 13:23:58,741 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for DecisionTreeClassifier classification model.


              precision    recall  f1-score   support

   Not Churn       0.89      0.67      0.76      1033
       Churn       0.45      0.77      0.57       374

    accuracy                           0.69      1407
   macro avg       0.67      0.72      0.67      1407
weighted avg       0.77      0.69      0.71      1407



In [26]:
metrics_RF=evaluate_model(fitted_pipe_RF,X_test,y_test)
print(metrics_RF['report'])

2026-06-04 13:26:18,887 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for RandomForestClassifier classification model.


              precision    recall  f1-score   support

   Not Churn       0.89      0.77      0.83      1033
       Churn       0.54      0.75      0.63       374

    accuracy                           0.76      1407
   macro avg       0.72      0.76      0.73      1407
weighted avg       0.80      0.76      0.77      1407

